## [E] POLS, FE, RE

In [1]:
import pandas as pd
import numpy as np
import os
import re

import statsmodels.api as sm
import scipy.stats as stats
from scipy.stats import chi2

from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

In [2]:
path = "data/processed_data/"
file_name = "03_pols_fe_re_gmm_winsorized_data.csv"

date_column = "date" 
stock_column = "stock"

# Set path
os.chdir("..")
print(os.getcwd())

/workspaces/master-thesis-submission/src


In [3]:
pols_fe_re_data = pd.read_csv(path + file_name)

In [4]:
pols_fe_re_data.head()

,date,stock,stock_return_quarterly,beta,size,value,profitability,investment,price_momentum,esg_score,esg_momentum,industry,country_of_headquarters
0,2016-06-30,A2.MI,0.030621,1.470557,22.009083,0.748281,0.079054,-0.033443,-0.036842,59.459100,-1.622974,Utilities,Italy
1,2016-06-30,AAL.L,0.250263,0.558792,23.225868,1.461129,-0.304424,0.000000,0.324333,81.434442,-7.452243,Basic Materials,United Kingdom
2,2016-06-30,AALB.AS,-0.112824,1.229154,21.818938,0.424145,0.194845,0.000000,0.014720,17.964003,5.607441,Industrials,Netherlands
3,2016-06-30,ABBN.S,0.029602,0.988284,24.365696,0.362603,0.200431,0.011075,0.041707,88.899243,-3.994408,Industrials,Switzerland
4,2016-06-30,ABDN.L,-0.217060,1.664075,22.662034,0.796699,0.150175,0.000000,-0.256856,74.424211,-0.391662,Financials,United Kingdom


## Analysis

In [5]:
def omitted_base_category(series):
    cats = sorted(series.dropna().astype(str).unique())
    return cats[0] if len(cats) else None, cats

In [6]:
def run_pols(df, include_time_dummies_pols_re=False):

    df = df.copy()
    df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

    # Set multi-index for panel data
    df = df.set_index([stock_column, date_column]).sort_index()

    # Define dependent and independent variables
    reg_vars = [
        "beta", 
        "size", 
        "value", 
        "profitability", 
        "investment",
        "price_momentum", 
        "esg_score", 
        "esg_momentum"
    ]

    base = df[reg_vars].copy()

    base_industry, industries = omitted_base_category(df["industry"])
    base_country, countries = omitted_base_category(df["country_of_headquarters"])

    print("Omitted base industry:", base_industry)
    print("Omitted base country:", base_country)

    industry_dummy = pd.get_dummies(df["industry"], drop_first=True, dtype=float)
    country_dummy = pd.get_dummies(df["country_of_headquarters"], drop_first=True, dtype=float)

    # Dependent variable
    y = df["stock_return_quarterly"]

    # Base X used for FE models (unchanged)
    X_base = pd.concat([base, industry_dummy, country_dummy], axis=1)
    X_base = sm.add_constant(X_base).astype(float)

    # Optional time dummies only for POLS and RE
    if include_time_dummies_pols_re:
        time_values = pd.Series(
            df.index.get_level_values(date_column),
            index=df.index,
            name="time_period"
        )
        
        time_dummy = pd.get_dummies(time_values, prefix="time", drop_first=True, dtype=float)

        X_pols_re = pd.concat([X_base, time_dummy], axis=1)
        X_pols_re = X_pols_re.astype(float)
    else:
        X_pols_re = X_base.copy()

    # Pooled OLS
    pooled_ols_model = PooledOLS(y, X_pols_re)
    pooled_ols_results = pooled_ols_model.fit(cov_type="clustered", cluster_entity=True)

    # Two-way Fixed Effects Model
    fe_model_two_way = PanelOLS(y, X_base, entity_effects=True, time_effects=True, drop_absorbed=True)
    fe_results_two_way = fe_model_two_way.fit(cov_type="clustered", cluster_entity=True)

    # Random Effects Model
    re_model = RandomEffects(y, X_pols_re)
    re_results = re_model.fit(cov_type="clustered", cluster_entity=True)
    re_theta = re_results.theta

    return pooled_ols_results, fe_results_two_way, re_results, re_theta

### Panel Regressions

In [7]:
pooled_ols_result, fe_results, re_result, re_theta = run_pols(pols_fe_re_data, include_time_dummies_pols_re=True)

Omitted base industry: Basic Materials
Omitted base country: Austria


/tmp/ipykernel_14438/2178028754.py:60: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Consumer Discretionary, Consumer Staples, Energy, Financials, Health Care, Industrials, Real Estate, Technology, Telecommunications, Utilities, Belgium, Bermuda, Cyprus, Czech Republic, Denmark, Faroe Islands, Finland, France, Germany, Hungary, Ireland, Isle of Man, Israel, Italy, Jersey, Kazakhstan, Luxembourg, Malta, Mexico, Netherlands, Norway, Poland, Portugal, South Africa, Spain, Sweden, Switzerland, United Kingdom, United States of America

  fe_results_two_way = fe_model_two_way.fit(cov_type="clustered", cluster_entity=True)


In [ ]:
def results_to_df(results):
    """ This function exports the pols, fe, and re results to xlsx """
    ci = results.conf_int()
    ci.columns = ["ci_lower", "ci_upper"]

    out = pd.concat(
        [
            results.params.rename("coef"),
            results.std_errors.rename("std_error"),
            results.tstats.rename("t_stat"),
            results.pvalues.rename("p_value"),
            ci,
        ],
        axis=1
    )
    out.index.name = "variable"
    return out

pooled_df = results_to_df(pooled_ols_result)
fe_df = results_to_df(fe_results)
re_df = results_to_df(re_result)

with pd.ExcelWriter(path + "05_panel_model_results.xlsx", engine="openpyxl") as writer:
    pooled_df.to_excel(writer, sheet_name="POLS")
    fe_df.to_excel(writer, sheet_name="FE")
    re_df.to_excel(writer, sheet_name="RE")

print("Results Exported to 05_panel_model_results.xlsx")

Results Exported to 05_panel_model_results.xlsx


In [10]:
pooled_ols_result

Dep. Variable:,stock_return_quarterly,R-squared:,0.4418
Estimator:,PooledOLS,R-squared (Between):,0.6510
No. Observations:,20184,R-squared (Within):,0.4323
Date:,"Fri, May 08 2026",R-squared (Overall):,0.4418
Time:,09:20:11,Log-likelihood,1.481e+04
Cov. Estimator:,Clustered,,
,,F-statistic:,189.40
Entities:,804,P-value,0.0000
Avg Obs:,25.104,Distribution:,"F(84,20099)"
Min Obs:,1.0000,,
Max Obs:,38.000,F-statistic (robust):,2.584e+16


In [11]:
fe_results

Dep. Variable:,stock_return_quarterly,R-squared:,0.2253
Estimator:,PanelOLS,R-squared (Between):,0.3246
No. Observations:,20184,R-squared (Within):,0.2007
Date:,"Fri, May 08 2026",R-squared (Overall):,0.1657
Time:,09:20:11,Log-likelihood,1.512e+04
Cov. Estimator:,Clustered,,
,,F-statistic:,702.97
Entities:,804,P-value,0.0000
Avg Obs:,25.104,Distribution:,"F(8,19335)"
Min Obs:,1.0000,,
Max Obs:,38.000,F-statistic (robust):,734.89


In [12]:
re_result

Dep. Variable:,stock_return_quarterly,R-squared:,0.4418
Estimator:,RandomEffects,R-squared (Between):,0.6510
No. Observations:,20184,R-squared (Within):,0.4323
Date:,"Fri, May 08 2026",R-squared (Overall):,0.4418
Time:,09:20:12,Log-likelihood,1.481e+04
Cov. Estimator:,Clustered,,
,,F-statistic:,189.40
Entities:,804,P-value,0.0000
Avg Obs:,25.104,Distribution:,"F(84,20099)"
Min Obs:,1.0000,,
Max Obs:,38.000,F-statistic (robust):,2.584e+16


### Hausman Test

In [23]:
def hausman_fe_re(fe_res, re_res, coef_names=None):
    """
        Source: https://github.com/bilkent-sna/negotiating-comfort-with-llm-agents/blob/main/analysis.ipynb
    """

    b_fe = fe_res.params
    b_re = re_res.params

    common = b_fe.index.intersection(b_re.index)

    if coef_names is not None:
        common = pd.Index([c for c in coef_names if c in common])

    diff = (b_fe.loc[common] - b_re.loc[common]).to_numpy()

    V_fe = fe_res.cov.loc[common, common].to_numpy()
    V_re = re_res.cov.loc[common, common].to_numpy()

    V_diff = V_fe - V_re

    stat = float(diff.T @ np.linalg.pinv(V_diff) @ diff)
    df = len(common)
    pval = 1.0 - chi2.cdf(stat, df)

    return {
        "stat": stat,
        "df": df,
        "pval": pval,
        "compared_coefficients": list(common),
    }

In [24]:
hausman_fe_re(fe_results, re_result)

{'stat': 57.92930583638603,
 'df': 9,
 'pval': 3.3533722465506344e-09,
 'compared_coefficients': ['const',
  'beta',
  'size',
  'value',
  'profitability',
  'investment',
  'price_momentum',
  'esg_score',
  'esg_momentum']}

### Random Effects Deep Dive

In [ ]:
def inspect_re_variance_components(df):

    # Source: https://bashtage.github.io/linearmodels/devel/_modules/linearmodels/panel/model.html

    # Match run_pols defaults
    use_lagged_regressors = False
    include_time_dummies_pols_re = True

    df = df.copy()
    df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

    # Keep only stocks with at least 2 observations
    counts = df.groupby(stock_column)[date_column].transform("count")
    df = df[counts >= 2]

    # Set panel index
    df = df.set_index([stock_column, date_column]).sort_index()

    # Same regressors as in run_pols
    reg_vars = [
        "beta", "size", "value", "profitability", "investment",
        "price_momentum", "esg_score", "esg_momentum"
    ]

    base = df[reg_vars].copy()

    if use_lagged_regressors:
        base = base.groupby(level=0).shift(1)

    industry_dummy = pd.get_dummies(df["industry"], drop_first=True, dtype=float)
    country_dummy = pd.get_dummies(df["country_of_headquarters"], drop_first=True, dtype=float)

    y = df["stock_return_quarterly"]

    X_base = pd.concat([base, industry_dummy, country_dummy], axis=1)
    X_base = sm.add_constant(X_base).astype(float)

    if include_time_dummies_pols_re:
        time_values = pd.Series(
            df.index.get_level_values(date_column),
            index=df.index,
            name="time_period"
        )
        time_dummy = pd.get_dummies(time_values, prefix="time", drop_first=True, dtype=float)
        X_pols_re = pd.concat([X_base, time_dummy], axis=1).astype(float)
    else:
        X_pols_re = X_base.copy()

    # Align and drop missing rows exactly before fitting
    valid = y.notna() & X_pols_re.notna().all(axis=1)
    y = y.loc[valid]
    X_pols_re = X_pols_re.loc[valid]

    # Fit RE model exactly as in run_pols
    re_model = RandomEffects(y, X_pols_re)
    re_results = re_model.fit(cov_type="clustered", cluster_entity=True)

    # --- Reconstruct sigma2_e and raw sigma2_u from the fitted model internals ---

    w = re_model.weights.values2d
    root_w = np.sqrt(w)

    # Weighted quasi-demeaned regression used for sigma2_e
    demeaned_dep = re_model.dependent.demean("entity", weights=re_model.weights)
    demeaned_exog = re_model.exog.demean("entity", weights=re_model.weights)

    y_within = demeaned_dep.values2d.copy()
    x_within = demeaned_exog.values2d.copy()

    if re_model.has_constant:
        w_sum = w.sum()
        y_gm = (w * re_model.dependent.values2d).sum(0) / w_sum
        x_gm = (w * re_model.exog.values2d).sum(0) / w_sum
        y_within = y_within + root_w * y_gm
        x_within = x_within + root_w * x_gm

    params_within = np.linalg.lstsq(x_within, y_within, rcond=None)[0]
    weps = y_within - x_within @ params_within

    # Between regression used for sigma2_u
    wybar = re_model.dependent.mean("entity", weights=re_model.weights)
    wxbar = re_model.exog.mean("entity", weights=re_model.weights)

    wybar_arr = np.asarray(wybar)
    wxbar_arr = np.asarray(wxbar)

    params_between = np.linalg.lstsq(wxbar_arr, wybar_arr, rcond=None)[0]
    wu = wybar_arr - wxbar_arr @ params_between

    # Dimensions used in linearmodels
    nobs = weps.shape[0]
    neffects = wu.shape[0]
    nvar = x_within.shape[1]

    sigma2_e_manual = float(np.squeeze(weps.T @ weps)) / (nobs - nvar - neffects + 1)

    t = np.asarray(re_model.dependent.count("entity"), dtype=float)
    t_bar = neffects / np.sum(1.0 / t)

    ssr_u = float(np.squeeze(wu.T @ wu))
    sigma2_u_raw = ssr_u / (neffects - nvar) - sigma2_e_manual / t_bar
    sigma2_u_after_clip = max(0.0, sigma2_u_raw)

    theta_manual = 1.0 - np.sqrt(
        sigma2_e_manual / (t * sigma2_u_after_clip + sigma2_e_manual)
    )

    theta_reported = re_results.theta["theta"].to_numpy()

    return {
        "sigma2_u_raw": sigma2_u_raw,
        "sigma2_u_after_clip": sigma2_u_after_clip,
        "sigma2_u_reported": float(re_results.variance_decomposition["Effects"]),
        "sigma2_e_manual": sigma2_e_manual,
        "sigma2_e_reported": float(re_results.variance_decomposition["Residual"]),
        "ssr_u": ssr_u,
        "t_bar": float(t_bar),
        "nobs": int(nobs),
        "neffects": int(neffects),
        "nvar": int(nvar),
        "all_theta_zero": bool((re_results.theta["theta"] == 0).all()),
        "theta_matches_manual": bool(np.allclose(theta_manual.ravel(), theta_reported)),
        "theta_summary": re_results.theta["theta"].describe(),
        "re_results": re_results,
    }

In [20]:
re_theta

,theta
stock,
1COVG.DE^L25,0.0
1U1.DE,0.0
A2.MI,0.0
AAAA.L^C21,0.0
AAK.ST,0.0
...,...
ZALG.DE,0.0
ZEG.L,0.0
ZELA.CO,0.0


In [21]:
print(re_result.variance_decomposition)

Effects                   0.00000
Residual                  0.01369
Percent due to Effects    0.00000
Name: Variance Decomposition, dtype: float64


In [22]:
inspect_re_variance_components(pols_fe_re_data)

{'sigma2_u_raw': -0.0003793778224851107,
 'sigma2_u_after_clip': 0.0,
 'sigma2_u_reported': 0.0,
 'sigma2_e_manual': 0.013690035158601564,
 'sigma2_e_reported': 0.013690035158601564,
 'ssr_u': 0.46980223066521015,
 't_bar': 13.138174268419885,
 'nobs': 20174,
 'neffects': 794,
 'nvar': 85,
 'all_theta_zero': True,
 'theta_matches_manual': True,
 'theta_summary': count    794.0
 mean       0.0
 std        0.0
 min        0.0
 25%        0.0
 50%        0.0
 75%        0.0
 max        0.0
 Name: theta, dtype: float64,
 're_results':                           RandomEffects Estimation Summary                          
 Dep. Variable:     stock_return_quarterly   R-squared:                        0.4414
 Estimator:                  RandomEffects   R-squared (Between):              0.6474
 No. Observations:                   20174   R-squared (Within):               0.4323
 Date:                    Fri, May 08 2026   R-squared (Overall):              0.4414
 Time:                            